# 16. Komparatív Többterületes Elemzés

**Cél:** a vizsgált terület (fő mintaterület) összehasonlítása a regisztrált kontroll- és benchmark-területekkel a közös, kanonikus mutatók mentén (kínálati méret, GIS-pontosság, medián fajlagos ár, vasúti immissziós hatás, állomáselérés).

---

### 📖 Módszertani megjegyzés
A területek összehasonlítása csak azonos definíciójú mutatók mentén értelmezhető — ehhez minden terület adata ugyanazon a kanonikus sémán és számítási láncon megy keresztül. A devizaeltérések miatt a fajlagos árak külön is értelmezendők.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import sys, os
from ingatlan_tdk._utils import *
setup_plotly()
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import pandas as pd
import numpy as np

# Területek konfigurációjának ellenőrzése
cfg = load_areas_config()
available_areas = list_available_areas()
print(f"Regisztrált és elérhető területek: {available_areas}")


### 1. Mintaterületek Leíró Összehasonlítása és Piaci Mérete

In [ ]:
# Területek betöltése és egyesítése komparatív elemzéshez
df_comp = load_all_areas_comparison(available_areas)
elado = df_comp[df_comp['listing_type'] == 'elado'].copy()

summary = elado.groupby('area_name').agg(
    lakasok_szama=('listing_id', 'count'),
    pontos_lakasok=('minta_garantalt_pontos', lambda x: int((x == 1).sum())),
    median_nm_ar=('nm_ar_huf', 'median'),
    atlag_ar_mft=('ar_millio_ft', 'mean'),
    atlag_terulet=('alapterulet_nm', 'mean')
).reset_index()

kpis = []
for _, r in summary.iterrows():
    kpis.append((f"{r['area_name']} Kínálat", f"{r['lakasok_szama']} db", f"Garantált pontos GIS: {r['pontos_lakasok']} db", "#1e3a8a"))
    kpis.append((f"{r['area_name']} Medián Ár", fmt_huf(r['median_nm_ar']), f"Átlagár: {r['atlag_ar_mft']:.1f} M Ft", "#059669"))

display(HTML(kpi_grid_html(kpis)))
display(summary.style.format({
    'lakasok_szama': '{:,} db',
    'pontos_lakasok': '{:,} db',
    'median_nm_ar': lambda v: fmt_huf(v),
    'atlag_ar_mft': '{:.1f} M Ft',
    'atlag_terulet': '{:.1f} m²'
}))


### 2. A Vasúti Immissziós Diszkont Összehasonlítása (0-150-300-500-1000-2000 m)
A legfontosabb ökonometriai kérdés: a vágányok közvetlen szomszédságában (<150m) fellépő árhátrány mekkora a kontroll területeken?


In [ ]:
immisszio_stats = elado[elado['vasut_zona'].notna()].groupby(['area_name', 'vasut_zona'], observed=False).agg(
    median_nm_ar=('nm_ar_huf', 'median'),
    darab=('listing_id', 'count')
).reset_index()

fig_imm = px.bar(
    immisszio_stats,
    x='vasut_zona',
    y='median_nm_ar',
    color='area_name',
    barmode='group',
    title='Medián Négyzetméterár a Szigorú Vasúti Immissziós Sávokban a Mintaterületeken',
    labels={'vasut_zona': 'Vasúti Immissziós Zóna', 'median_nm_ar': 'Medián Ár/m² (Ft)', 'area_name': 'Vizsgált Terület'},
    color_discrete_sequence=['#2563eb', '#059669', '#d97706', '#dc2626']
)
fig_imm.update_layout(xaxis_tickangle=-25, height=520)
fig_imm.show()


### 3. Állomáskörnyéki Gyalogos Elérhetőség (TOD Prémium) az Izokrón Sávokban (375m, 750m, 1125m)

In [ ]:
izokron_lista = []
for p, m_dist in [(5, '≤375 m (5 perc)'), (10, '≤750 m (10 perc)'), (15, '≤1125 m (15 perc)')]:
    col = f'vasut_{p}p_seta'
    if col in elado.columns:
        for area_name, grp in elado.groupby('area_name'):
            minta = grp[grp[col] == 1]
            if len(minta) > 0:
                izokron_lista.append({
                    'Terület': area_name,
                    'Gyalogos Izokrón': m_dist,
                    'Lakásszám (db)': len(minta),
                    'Medián Ár/m²': minta['nm_ar_huf'].median(),
                    'Átlagár (M Ft)': minta['ar_millio_ft'].mean()
                })

df_izo_comp = pd.DataFrame(izokron_lista)

fig_izo = px.bar(
    df_izo_comp,
    x='Gyalogos Izokrón',
    y='Medián Ár/m²',
    color='Terület',
    barmode='group',
    title='Állomáskörnyéki TOD Gyalogos Elérhetőség (375m / 750m / 1125m) a Mintaterületeken',
    labels={'Gyalogos Izokrón': 'Nemzetközi Gyalogos Izokrón (1.25 m/s)', 'Medián Ár/m²': 'Medián Ár/m² (Ft)'},
    color_discrete_sequence=['#2563eb', '#059669', '#d97706', '#dc2626']
)
fig_izo.update_layout(height=480)
fig_izo.show()


### 💡 4. Tudományos Szintézis és TDK Eredmények
- **Robusztusság igazolása**: A kontroll területek bevonásával számszerűsíthető a budapesti vasúti zöldfolyosók és barnamezők általános piaci reakciója.
- **Várospolitikai tanulság**: Az immissziós sávok (<150m) zajvédelmi rehabilitációja és a 375m-es állomáskörzeti gyalogos infrastruktúra-fejlesztés egyetemes módon szabadítja fel a látens ingatlanpiaci tőketartalékot (Rent Gap) mind a fővárosi, mind a kerületi szinten.
